# Embedding mAP: all-cells vs top-predictive (Figure 3D)

Phase-only CellDINO embedding mAP for two readouts, before and after set-accuracy distillation:

- **Gene KO** — phenotypic distinctiveness (mAP over per-gene queries, n = 1,001).
- **Protein complex** — EBI consistency (mAP over per-complex queries, n = 98).

`all cells` = plain CellDINO on every cell (`.../phase_only/fixed_80%/cosine/`). `top-predictive` = same embedding weighted by the multibag SHAP-ranked top cells per gene / complex (`.../attention/v5m_gko_cutoff_20k/` for gene-level, `.../attention/v5m_ebionly_cutoff_20k/` for complex). Distillation lifts the median from 0.456 → 1.000 (gene) and 0.531 → 1.000 (complex).

Inputs are the per-gene / per-complex `mean_average_precision` CSVs from the PCA-optimization pipeline, one row per query, read from `../../../data/figures/figure_3/`:

| file | source metric |
| --- | --- |
| `figure_3d_embedding_map_phase_distinct_allcells.csv` | phase_only baseline — phenotypic distinctiveness (gene KO) |
| `figure_3d_embedding_map_phase_distinct_topcells.csv` | v5m_gko_cutoff_20k distilled — phenotypic distinctiveness (gene KO) |
| `figure_3d_embedding_map_phase_ebi_allcells.csv` | phase_only baseline — EBI consistency (protein complex) |
| `figure_3d_embedding_map_phase_ebi_topcells.csv` | v5m_ebionly_cutoff_20k distilled — EBI consistency (protein complex) |

> **Note.** The complex CSVs carry 98 queries (`complex_num` 1–98), one fewer than the
> 99 complexes classified in Figure 3B. In the `top-predictive` condition the mAP is
> exactly 1.000 for 78% of genes and 68% of complexes, which is what pins both medians
> at 1.000.


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator

# Keep text editable in Illustrator.
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURES_DIR = Path("../../../output/figure_3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data paths

Per-query mAP CSVs (one row per gene / complex), read from the central
figure-data directory `../../../data/figures/figure_3/`.

In [ ]:
FIGURE_DATA = Path("../../../data/figures/figure_3")

GROUPS = [
    ("Gene\nKO",         "figure_3d_embedding_map_phase_distinct_allcells.csv",
                         "figure_3d_embedding_map_phase_distinct_topcells.csv"),
    ("Protein\ncomplex", "figure_3d_embedding_map_phase_ebi_allcells.csv",
                         "figure_3d_embedding_map_phase_ebi_topcells.csv"),
]

## Configuration

Two violins per group (all cells vs top-predictive); colours match the paper draft (blue = all cells, green = top-predictive).

In [ ]:
COLOR_ALL_CELLS      = "#7B9FC6"  # muted blue
COLOR_TOP_PREDICTIVE = "#5CA976"  # muted green

VIOLIN_WIDTH = 0.45
WITHIN_GAP   = 0.28   # offset between the two violins within a group
GROUP_GAP    = 1.4    # spacing between the two groups

## Load

Each CSV holds one `mean_average_precision` per query (per-gene for distinctiveness, per-complex for EBI). `dropna` guards against queries with too few members to score; none of the four current CSVs contain any, so all 1,001 / 98 queries are used.

In [ ]:
def load_map(csv: str) -> np.ndarray:
    return pd.read_csv(FIGURE_DATA / csv)["mean_average_precision"].dropna().to_numpy()

data = []
for label, all_csv, top_csv in GROUPS:
    data.append((label, load_map(all_csv), load_map(top_csv)))

for label, a, t in data:
    lbl = label.replace("\n", " ")
    print(f"{lbl:>16}: all cells n={len(a):>4} median={np.median(a):.3f} | "
          f"top-predictive n={len(t):>4} median={np.median(t):.3f}")

## Figure 3D

Two groups, two violins each, median as a horizontal black bar. Saves an SVG (paper) + PNG.

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 5.0))
centers = [gi * GROUP_GAP for gi in range(len(data))]

for center, (_label, all_vals, top_vals) in zip(centers, data):
    for pos_off, vals, color in [
        (-WITHIN_GAP, all_vals, COLOR_ALL_CELLS),
        ( WITHIN_GAP, top_vals, COLOR_TOP_PREDICTIVE),
    ]:
        pos = center + pos_off
        vp = ax.violinplot(vals, positions=[pos], widths=VIOLIN_WIDTH,
                            showextrema=False, showmedians=False)
        for body in vp["bodies"]:
            body.set_facecolor(color)
            body.set_edgecolor("none")
            body.set_alpha(0.9)
        med = float(np.median(vals))
        ax.hlines(med, pos - VIOLIN_WIDTH / 2, pos + VIOLIN_WIDTH / 2,
                   color="black", lw=2.5, zorder=5)

ax.set_xticks(centers)
ax.set_xticklabels([g[0] for g in GROUPS], fontsize=11)
ax.set_ylim(0.0, 1.05)
ax.yaxis.set_major_locator(MultipleLocator(0.2))
ax.set_ylabel("mAP score", fontsize=11)
ax.tick_params(axis="y", labelsize=10)
ax.tick_params(axis="x", length=0)
ax.grid(True, axis="y", alpha=0.25)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

legend_handles = [
    Patch(facecolor=COLOR_ALL_CELLS,      label="all cells"),
    Patch(facecolor=COLOR_TOP_PREDICTIVE, label="top-predictive"),
]
ax.legend(handles=legend_handles, loc="upper left",
           bbox_to_anchor=(0.0, 1.18), ncols=1, frameon=False, fontsize=10)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "embedding_map_violin.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "embedding_map_violin.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Per (group, condition) median / mean / n — the numbers behind the violins.

In [ ]:
rows = []
for label, all_vals, top_vals in data:
    lbl = label.replace("\n", " ")
    for cond, vals in [("all cells", all_vals), ("top-predictive", top_vals)]:
        rows.append({"group": lbl, "condition": cond, "n": int(len(vals)),
                     "median": float(np.median(vals)),
                     "mean":   float(np.mean(vals))})

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "embedding_map_violin_summary.csv", index=False)
summary